Today we would like to learn skills to dissect a dataset. We consider the classic [Iris dataset](https://en.wikipedia.org/wiki/Iris_flower_data_set), which was collected by the botanist Edgar Anderson and made famous by the pioneer of statistical science himself, [RA Fisher](https://onlinelibrary.wiley.com/doi/10.1111/j.1469-1809.1936.tb02137.x), in 1936.

In [ ]:
import numpy as np

import pandas as pd
import sklearn

First we load the dataset from scikit learn. Setting the `as_frame` argument will lead to `data` consisting of `pandas` dataframes, an object we will learn how to handle today.

In [ ]:
data = sklearn.datasets.load_iris(as_frame=True)
print(type(data))

Looking at `data` more closely, we see that it is a sklearn's native `bunch` object, but it seems to build upon the python dictionary. Let's consider its keys 

In [ ]:
print(data.keys())

In [ ]:
for key, value in data.items():
    print(f"key {key}", type(value).__name__, sep=' has type: ')

Let's look at the data's `frame` which nicely illustrates what kind of data we're actually dealing with.

In [ ]:
df = data["frame"].copy()
df

We see 150 rows of 5 columns and an additional running index. The 5 columns split into 4 features (sepal length, sepal width, petal length, petal width all in cm) and one target. As you might have guessed, the idea is to predict the target from its 4 features. 

Let's get the target names from the data bunch object


In [ ]:
print(data["target_names"])

To get a better idea, here is a picture of the three different flowers

![iris types](../assets/iris.png)
 
 [source](https://www.kaggle.com/code/sunaysawant/iris-eda)



Let's consider the dataframe a bit more closely (you can also consider a short pandas introduction [here](https://pandas.pydata.org/docs/user_guide/10min.html)). What are the types of individual columns?

In [ ]:
sepal_length = df["sepal length (cm)"]
type(sepal_length)

Roughly we can think of a `Series` as a (numpy) array together with a label (in pandas lingo referred to as `index`). Use the ever helpful `help` on `pd.Series` to get an idea

In [ ]:
help(pd.Series)

We can slice a pandas Series just like we would a numpy array. Careful with single elements, though: `series[1]` does not return the element at *position* 1, but the element whose *label* (index) is 1. Here, labels and positions coincide, as the index runs from 0 to 149. After selecting a subset of the rows, e.g. only one class, they no longer do. To access an element by its position, use `.iloc`.

In [ ]:
print(sepal_length[0:5])
print(f"{type(sepal_length[0:5]).__name__ = }")  # slices will again be a series
print()

print(f"{sepal_length[1] = }")  # label 1
print(f"{sepal_length.iloc[1] = }")  # position 1

Or can extract from them a list or numpy array

In [ ]:
sepal_length_list = sepal_length.to_list()
print(f"{type(sepal_length_list).__name__ = }")


sepal_length_np = sepal_length.to_numpy()
print(f"{type(sepal_length_np).__name__ = }")

Reconsidering our data in the dataframe, it is a bit clunky to have "(cm)" in the columns when working with them. Since all values are on the same scale, we can rename them without significant loss of information

In [ ]:
print(f"before \n {df.head()}")
old_names = [name for name in df.columns.to_list() if "(cm)" in name]
name_map = {old_name: old_name.removesuffix(" (cm)") for old_name in old_names}
df = df.rename(name_map, axis='columns')
print(f" \n after \n {df.head()}")

Say we wanted to extend our dataframe with new columns that are combinations of the old ones. For instance, we can compute an approximate surface area of the sepals and petals. Let's assume the petals can be approximated as rectangles (A = width * length), the sepals as triangles (A = 1/2 * width * length). 

We can assign new columns of the dataframe, just as we would introduce new key-value pairs in a dictionary. As input we can simply multiply the corresponding series

In [ ]:
df["petal area"] = df["petal width"] * df["petal length"]
df

Analogously, add the column for sepal area

In [ ]:
df["sepal area"] = ...
df

Next, we want to consider the individual classes (targets). For that we can index all rows with a certain column value for `target` by first creating an appropriate boolean mask.

In [ ]:
mask_0 = df["target"] == 0  # This creates a boolean mask with values being True, if the value is 0, False otherwise
print(mask_0)
# Interestingly the mask itself is a Series
print(f"{type(mask_0) = }")

By use of the mask, we create a sub dataframe only consisting of class 0

In [ ]:
df_0 = df[mask_0]
assert (df_0["target"] == 0).all()

Do the same for class 1 and 2

In [ ]:
df_1 = ...
assert (df_1["target"] == 1).all()

df_2 = ...
assert (df_2["target"] == 2).all()

Compare the values of the different classes. Pandas Series has most of the methods and attributes that numpy arrays have. Use that to compare the mean and standard deviations of the different dataframes. 

Maybe you could even call the methods on the dataframes itself. 

In [ ]:
# TODO: Adapt the print statements to actually show some stats
print("simple stats of class 0")

print("simple stats of class 1")

print("simple stats of class 2")

Let's create a new dataframe. The rows shall represent classes and as columns we want to gather some statistics

In [ ]:
dfs = [df_0, df_1, df_2]
mean_dict = {
    'petal area mean': [frame["petal area"].mean() for frame in dfs],
    'sepal area mean': [frame["sepal area"].mean() for frame in dfs]
}
meta_df = pd.DataFrame(mean_dict)
meta_df

Likewise, add the corresponding standard deviation

In [ ]:
meta_df["petal area std"] = ...
meta_df["sepal area std"] = ...
meta_df

Very good, you've gathered some pandas basics. This makes life easy for working with tabular data.

To get an even better picture of your data, we want to cover basic visualization next. An essential tool for this is the [matplotlib library](https://matplotlib.org/).  You'll get a more thorough introduction [here](https://matplotlib.org/stable/users/explain/quick_start.html#quick-start).

Remember the next import line, as you will see/need it (almost) everywhere where data in python is visualized.

In [ ]:
import matplotlib.pyplot as plt

We can use the simple `plt.plot(x, y)` function to connect points in a list (or numpy array or other sequences)

In [ ]:
x = [-3, -2, -1, 0]
y = [0, 1, 2, 3]
plt.plot(x, y)

`plot`'s first argument `x` is optional. If you do not specify it, an index list `[0, ..., len(y) - 1]` will be used.

In [ ]:
plt.plot(y)  # note the different x tick labels

Let's use the visualization on our data right away and vizualize sepal width on the horizontal axis, sepal length on the vertical axis.

In [ ]:
plt.plot(df["sepal width"], df["sepal length"])

Now you should see some overlapping scribblings on the plot. `plt.plot()` by default connects the individual data points. One can use some workarounds to make `plot` not show lines connecting the points but this would not be its dedicated use-case. It is, however, the dedicated use case for a scatter plot.

In [ ]:
plt.scatter(df["sepal width"], df["sepal length"], s=15)  # s controls the marker size

plt.xlabel("sepal width")  # specifying an x-label
plt.ylabel("sepal length") # specifying a y-label

Note that `plt.scatter` needs the two arguments `x` and `y` and won't work when passing one argument only.

Now consider the scatter plot. Over the whole dataset, sepal width and length hardly seem to be related (their correlation coefficient is about $-0.12$). One might, however, reason that the data splits into two clusters (upper left and lower right). To examine this claim, we want each marker to reflect its class. One way to do so, is by color, as `plt.scatter` takes a `c` argument which can for instance be a single string (e.g. `red`) or an iterable that matches x and y values in shape so that for every data point there exists one color entry. We could pass the dataframe's `target` column, which is an iterable, as the color argument, but we'll go for a less efficient but clearer approach.

First we map every target class to a specific color

In [ ]:
color_map = {
    0: "red",
    1: "green",
    2: "blue"
}

And then we iterate over the target values. We could do so by using the individual class dataframes we extracted earlier (`df_0`, ...) but just to make sure we're using the same data during one iteration, we'll mask inside the loop.

In [ ]:
for class_int, class_color in color_map.items():
    class_df = df[df["target"] == class_int]
    plt.scatter(class_df["sepal width"], class_df["sepal length"], s=15, c=class_color)
plt.xlabel("sepal width")  # specifying an x-label
plt.ylabel("sepal length") # specifying a y-label

Great, we can indeed see that the lower right cluster consists of one class only while the upper left one is a mix. Note also that *within* each class, sepal width and length are clearly positively correlated (correlation coefficients between 0.46 and 0.74), although they hardly are over the whole dataset. Looking at the classes separately can thus reveal relationships that the whole dataset hides.

For a user (or for ourselves later) that might not have seen the definition of the color map, we should convey its information inside the plot. Thus, we add a *legend*.

In [ ]:
for class_int, class_color in color_map.items():
    class_df = df[df["target"] == class_int]
    plt.scatter(class_df["sepal width"], class_df["sepal length"], s=15, c=class_color)

plt.xlabel("sepal width")  # specifying an x-label
plt.ylabel("sepal length") # specifying a y-label
plt.legend(color_map)

It would even be nicer to directly display the name of the flower instead of its target integers. 

Create the same plot from above, but have the legend display a mapping from color to flower name. Start from the given target integer to target name mapping

In [ ]:
int_to_name = {i: str(target_name) for i, target_name in enumerate(data.target_names)}
# TODO: adapt the above plot, showing target names instead of their integer representation

The previous plots indicate, that one class of the Iris dataset is *linearly* separable from the other two. 

Check if this remains true if you consider petal instead of sepal quantities. 

Plot the last scatter plot above with petal values

In [ ]:
# TODO: scatter plot of petal width vs. petal length, with the flower names in the legend

To slowly move towards learning a model, we want to split our data into train, validation and test split. Since sklearn does not have a dedicated train-val-test but only a train-test split function, we'll just use the function twice to create a 80:10:10 split.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_notrain, y_train, y_notrain = train_test_split(data.data, data.target, test_size = 0.2)
X_val, X_test, y_val, y_test = train_test_split(X_notrain, y_notrain, test_size = 0.5)

Is this a good split? What should we be worried about?